<a href="https://colab.research.google.com/github/isaacadebayo/Agentic_projects/blob/main/GRADIO_Multimodal_Chatbot_Longevity_COMPLETE_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installing Packages on Colab

In [1]:
%%time
! pip install --upgrade gradio
!pip install -q langchain langchain-openai faiss-cpu pypdf
!pip install langchain langchain-community
!pip install langchain langchain-huggingface
! pip install bs4
! pip install langchain
! pip install langchain-core
! pip install openai
! pip install -q google-genai openai
! pip install -q langgraph google-genai
! pip install sentence-transformers
! pip install edge-tts
! pip install mlflow
! pip install dvc-gs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: starlette
    Found existing installation: starlette 0.52.1
    Uninstalling starlette-0.52.1:
      Successfully uninstalled starlette-0.52.1
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.50.0
    Uninstalling gradio-5.50.0:
      Successfully uninstalled gradio-5.50.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.3.1 whi

#Importing Packages on Colab

In [2]:
%%time
import gradio as gr
import getpass
import os
import bs4
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from openai import OpenAI
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing_extensions import List, TypedDict
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from google import genai
from google.genai import types
from google.colab import drive

<timed exec>:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.


CPU times: user 28.9 s, sys: 5.26 s, total: 34.1 s
Wall time: 45.1 s


# Importing Dataset from Google Drive

In [5]:
%%time
from langchain_community.document_loaders import PyPDFLoader
drive.mount('/content/drive')

dataset = '/content/drive/MyDrive/longevity.pdf'

loader = PyPDFLoader(dataset)
docs = loader.load()
# Join the page content from the loaded documents into a single string for test_chatbot
longevity_chatbot = "\n\n".join([doc.page_content for doc in docs])

Mounted at /content/drive


CPU times: user 3.48 s, sys: 180 ms, total: 3.66 s
Wall time: 30.6 s


In [6]:
print(longevity_chatbot)

Visualizing and Predic0ng trend in global Life Expectancy from 2001 to 2019  1st Adebayo 2nd Azubuike 3rd Maddox-McGhee Schools of Applied Computa3onal Sci Schools of Applied Computa3onal Sci Schools of Applied Computa3onal Sci  Meharry Medical College Meharry Medical College Meharry Medical College  Nashville, USA Nashville, USA Nashville, USA  isaac.adebayo@mmc.edu charnisha.azubuike@mmc.edu michella.maddoxmcghee@mmc.edu Abstract—What factors drive longevity. Why are some island na6ons living longer than some developed na6ons. Longevity and life expectancy is a core issue that touches on the fundamental of livelihood, it drives immigra6on and emigra6on for a search of be=er quality of life, longevity contributes to brain drain of a region in search for a be=er opportunity to maximize the quality of a individual or family life. This project aims to explore this issue thoroughly by analzying the data from world bank via kaggle. The goal is to visualize which factors inﬂuences or drive 

In [ ]:
!pip install mlflow cryptography==43.0.3 --ignore-installed blinker

In [7]:
import mlflow
import mlflow.langchain
import mlflow.openai
# Define a wrapper function to integrate MLflow tracking for each chatbot interaction
def mlflow_tracked_ask_question(question: str, history=None) -> str:
    # Start an MLflow run for each interaction
    with mlflow.start_run(run_name=f"Query: {question[:50]}...") as run:
        mlflow.log_param("user_question", question)

        # Log chat history as a parameter or artifact (as JSON string)
        history_json = json.dumps(history if history is not None else [], indent=2)
        mlflow.log_text(history_json, "chat_history_before_query.json")
        mlflow.log_param("chat_history_length", len(history) if history else 0)

        # Log prompt templates as artifacts
        mlflow.log_text(chatbot_prompt.format(question="<QUESTION_PLACEHOLDER>", context="<CONTEXT_PLACEHOLDER>"), "chatbot_prompt_template.txt")
        mlflow.log_text(condense_question_prompt.format(question="<QUESTION_PLACEHOLDER>", chat_history=[]), "condense_question_prompt_template.txt")

        # Log LLM and Embedding model details
        mlflow.log_param("llm_model_name", llm.model_name) # Assuming llm is a ChatOpenAI instance
        mlflow.log_param("llm_temperature", llm.temperature)
        mlflow.log_param("llm_max_tokens", llm.max_tokens)
        mlflow.log_param("embedding_model_name", embeddings.model) # Assuming embeddings is an OpenAIEmbeddings instance
        mlflow.log_param("vector_store_type", "FAISS")

        # Call the original ask_question function to get the RAG answer
        rag_answer = ask_question(question, history)

        mlflow.log_param("bot_response", rag_answer)

        # Log input/output character counts as a proxy for token usage
        mlflow.log_metric("input_char_count", len(question))
        mlflow.log_metric("output_char_count", len(rag_answer))

        # Log response audio if available (only for voice_chat_handler)
        # Note: 'response.mp3' might not exist for text queries or if TTS fails
        try:
            if os.path.exists("response.mp3"):
                mlflow.log_artifact("response.mp3", "bot_audio_response")
        except Exception as e:
            print(f"Warning: Could not log response.mp3: {e}")

        return rag_answer

print("MLflow wrapper function 'mlflow_tracked_ask_question' defined.")

MLflow wrapper function 'mlflow_tracked_ask_question' defined.


## MLflow Tracking for RAG Chatbot

This section integrates MLflow to track the performance and components of the RAG chatbot. Each user interaction will create a new MLflow run, logging details such as the input question, the bot's response, prompt templates, and characteristics of the models and vector store used.

This allows for:
*   **Model Versioning**: Tracking which `llm` and `embedding` models are used.
*   **Prompt Versioning**: Logging the system and condensing prompts.
*   **Artifacts**: Storing conversation details and potential audio responses.
*   **Metrics**: Estimating token usage (via character count for simplicity).
*   **Experimentation**: Comparing different interactions or configurations.

In [8]:
'''import mlflow
import mlflow.langchain
import mlflow.openai
import os
import openai
import json

# Set the MLflow tracking URI to connect to the server started by ngrok
# The ngrok setup usually exposes localhost:5000
mlflow.set_tracking_uri("http://localhost:5000")

# Set an experiment name for better organization in the MLflow UI
mlflow.set_experiment("RAG Chatbot Interactions")
mlflow.openai.autolog()

print("MLflow tracking setup complete.")'''

'import mlflow\nimport mlflow.langchain\nimport mlflow.openai\nimport os\nimport openai\nimport json\n\n# Set the MLflow tracking URI to connect to the server started by ngrok\n# The ngrok setup usually exposes localhost:5000\nmlflow.set_tracking_uri("http://localhost:5000")\n\n# Set an experiment name for better organization in the MLflow UI\nmlflow.set_experiment("RAG Chatbot Interactions")\nmlflow.openai.autolog()\n\nprint("MLflow tracking setup complete.")'

Going forward change the dataset and import the right package

In [9]:
!pip install pyngrok

In [10]:
!pip install dvc[gdrive]

In [11]:
# Replace with your GitHub email and username
!git config --global user.email "isaac_adebayojr@gmail.com"
!git config --global user.name "isaacadebayo"

print("Git user configured.")

Git user configured.


In [12]:
# Store your GitHub PAT securely in Colab Secrets
# Click on the '🔑' icon on the left panel, add a new secret named 'GH_TOKEN' and paste your PAT there.
from google.colab import userdata
import os

# Get the PAT from Colab secrets
GH_TOKEN = userdata.get('GH_TOKEN')

# Set it as an environment variable (optional, but good practice if cloning private repos)
os.environ['GH_TOKEN'] = GH_TOKEN

print("GitHub Token retrieved. You can now use it for cloning private repositories.")

GitHub Token retrieved. You can now use it for cloning private repositories.


In [13]:
# Replace 'your_username' and 'your_repository' with your actual GitHub details.
# For a public repository:
!git clone https://github.com/isaacadebayo/Predictive-Analytics-Public-Datasets.git

# For a private repository using PAT:
# This uses the GH_TOKEN environment variable
!git clone https://{GH_TOKEN}@github.com/isaacadebayo/Predictive-Analytics-Public-Datasets

# Change into your repository directory to perform Git operations
%cd Predictive-Analytics-Public-Datasets

print("Repository cloned. You are now in the repository directory.")

Cloning into 'Predictive-Analytics-Public-Datasets'...
remote: Enumerating objects: 244, done.
remote: Counting objects: 100% (106/106), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 244 (delta 37), reused 35 (delta 4), pack-reused 138 (from 1)
Receiving objects: 100% (244/244), 30.32 MiB | 19.01 MiB/s, done.
Resolving deltas: 100% (110/110), done.
fatal: destination path 'Predictive-Analytics-Public-Datasets' already exists and is not an empty directory.
/content/Predictive-Analytics-Public-Datasets
Repository cloned. You are now in the repository directory.


In [14]:
# Authenticate Google Cloud for DVC to access GCS
# This will open a browser window for authentication.
# Follow the instructions to get an authorization code and paste it back.
!gcloud auth application-default login

print("Google Cloud authentication complete. You can now retry the DVC push.")

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=NqWEqjEuKZWBQDxVwZ6B5cQmdgCcKK&prompt=consent&token_usage=remote&access_type=offline&code_challenge=Ca5uDZ5aC2UyfrgTLBLVQrGJ8YA4C1NIRZNl7NwZD44&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0AdkVLPwEajLGQZtmVUZj2IlTUrrLgK6-z1BObsURNONputgArbl7rDFA9NbEfYZrqJXh6Q

Credentials saved to file: [/content/.config/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).
Ca

In [15]:
# Initialize DVC within the current directory (which is the cloned Git repo)
# !dvc init

# Configure Google Drive or cloud storage as a remote storage (replace 'dvc_storage' with your desired folder name in Google Drive)
# !dvc remote add -d gdrive_remote gdrive://MyDrive/dvc_storage
# !dvc remote add -d gcs_remote gs://my-agentic-chatbot-data-lake

# Create a data directory within the Git repository if it doesn't exist
!mkdir -p data

# Copy the data file into the Git repository's data directory
!cp /content/drive/MyDrive/longevity.pdf data/longevity.pdf

# Add a data file to DVC. This will create a .dvc file at data/longevity.pdf.dvc
!dvc add data/longevity.pdf

# Ensure mlflow.db and proxy.py are ignored by Git
# Append to the main .gitignore in the repo root, creating it if it doesn't exist
# Check if .gitignore exists, create if not, then append
gitignore_path = ".gitignore"
with open(gitignore_path, "a+") as f:
    f.seek(0)
    existing_content = f.read()
    if "mlflow.db" not in existing_content:
        f.write("\nmlflow.db\n")
    if "proxy.py" not in existing_content:
        f.write("proxy.py\n")

# Stage DVC file and any modified .gitignore files (both root and data folder)
!git add data/longevity.pdf.dvc
!git add .gitignore  # Stage the root .gitignore
!git add data/.gitignore # Stage data/.gitignore which DVC might have modified

# Commit the staged changes
!git commit -m "Add longevity.pdf DVC file and update gitignore"

# Push the DVC-tracked data to the remote (Google Drive / GCS)
!dvc push

print("DVC setup complete. Your data file is now versioned with DVC and stored in Google Drive.")
print("You can push/pull changes using: !dvc push and !dvc pull")

⠋ Checking graph
Adding...:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
!
          |0.00 [00:00,     ?file/s]
                                    
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
Adding data/longevity.pdf to cache:   0% 0/1 [00:00<?, ?file/s]
Adding data/longevity.pdf to cache:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
                                                                           
  0% 0/1 [00:00<?, ?files/s]
  0% 0/1 [00:00<?, ?files/s{'info': ''}]
Adding...: 100% 1/1 [00:00<00:00, 14.21file/s{'info': ''}]

To track the changes with git, run:

	git add data/longevity.pdf.dvc

To enable auto staging, run:

	dvc config core.autostage true
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Pushing
!
  0% |          |0/? [00:00<?,    ?files/s]/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user cre

### Authenticate Google Cloud for DVC with GCS

To allow DVC to push data to your Google Cloud Storage bucket, you need to authenticate your Google Cloud account. This will grant the necessary permissions for DVC to interact with GCS.

Run the following cell and follow the instructions to authenticate:

1.  Click the link generated in the output.
2.  Select your Google account.
3.  Click 'Allow' to grant access.
4.  Copy the authorization code provided.
5.  Paste the authorization code back into the input field in the Colab output and press Enter.

In [16]:
# Stage all changes in the current directory
!git add .

# Commit the staged changes with a message
# Replace 'Your insightful commit message' with a description of your changes
!git commit -m "Testing commit from colab to github"

# Push the changes to the 'main' branch of your remote repository
# This uses the GH_TOKEN environment variable for authentication
!git push https://{GH_TOKEN}@github.com/isaacadebayo/Predictive-Analytics-Public-Datasets.git main

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date


In [69]:
print('Contents of the cloned repository:')
!ls -F

Contents of the cloned repository:
 Apple_Stock_prediction.ipynb
 Apple_Stock_prediction_timeseries_split_fold4.ipynb
 Apple_Stock_prediction_timeseries_split.ipynb
 data/
'Fraud_detection_highEvalMatrix_(1).ipynb'
 Fraud_detection_highEvalMatrix_FINAL.ipynb
 Fraud_detection_highEvalMatrix.ipynb
 Fraud_detection.ipynb
'GRADIO_Multimodal_Chatbot_Longevity_ No_API_KEY.ipynb'
 Image_Generation_using_gpt_image_1.ipynb
 mlartifacts/
 mlflow.db
 Model_finetuning_mistral_ai.ipynb
 project/
 proxy.py
 RAG_Finanicial_Audio_Chatbot_Original_No_API_KEY.ipynb
 RAG_Finanicial_Chatbot_Original_NoAPI_Key.ipynb
 README.md
 response.mp3
 Time_Series_Financial_NYSE.ipynb
 Unsupervised_learning_Clustering_Credit_Card_Analysis.ipynb
 Visual_Transformer_and_CNN.ipynb
 visual_transformer_and_cnn.py
 WHISPER_Audio_Transformer.ipynb


## Using Ngrok for MLFlow log

In [158]:
import subprocess, time, requests, os, re
from pyngrok import ngrok
from google.colab import userdata

# ── 1. Kill everything ────────────────────────────────────────────
subprocess.run("pkill -f 'mlflow server'", shell=True)
subprocess.run("pkill -f proxy.py", shell=True)
subprocess.run("pkill -f cloudflared", shell=True)
time.sleep(3)

# ── 2. Start MLflow ───────────────────────────────────────────────
env = os.environ.copy()
env["GUNICORN_CMD_ARGS"] = "--forwarded-allow-ips='*'"

subprocess.Popen(
    ["mlflow", "server",
     "--backend-store-uri", "sqlite:////content/Predictive-Analytics-Public-Datasets/mlflow.db",
     "--serve-artifacts",
     "--host", "0.0.0.0",
     "--port", "5000",
     "--cors-allowed-origins", "https://ia-mlflow-dashboard.ngrok.app"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
    cwd="/content/Predictive-Analytics-Public-Datasets",
    env=env
)

print("Waiting for MLflow...")
for i in range(30):
    try:
        r = requests.get("http://localhost:5000/health", timeout=3)
        if r.status_code == 200:
            print(f"MLflow ready")
            break
    except Exception:
        pass
    time.sleep(2)

# ── 3. Write proxy (rewrites Host header to localhost:5000) ────────
proxy_code = """
import http.server
import http.client
import urllib.parse

class Proxy(http.server.BaseHTTPRequestHandler):
    def _proxy(self, body=None):
        parsed = urllib.parse.urlparse(f"http://localhost:5000{self.path}")

        conn = http.client.HTTPConnection("localhost", 5000, timeout=30)

        headers = dict(self.headers)
        headers["Host"] = "localhost:5000"
        headers["X-Forwarded-Host"] = "localhost:5000"
        headers["X-Forwarded-Proto"] = "http"
        headers.pop("Transfer-Encoding", None)
        headers.pop("transfer-encoding", None)

        try:
            conn.request(self.command, parsed.path + (f"?{parsed.query}" if parsed.query else ""), body=body, headers=headers)
            resp = conn.getresponse()

            self.send_response(resp.status)

            for k, v in resp.getheaders():
                if k.lower() not in ("transfer-encoding", "connection", "keep-alive"):
                    self.send_header(k, v)
            self.send_header("Access-Control-Allow-Origin", "*")
            self.send_header("Access-Control-Allow-Headers", "*")
            self.end_headers()

            # Stream response in chunks — fixes chart data loading
            while True:
                chunk = resp.read(8192)
                if not chunk:
                    break
                self.wfile.write(chunk)
            self.wfile.flush()

        except Exception as e:
            self.send_response(502)
            self.end_headers()
            self.wfile.write(str(e).encode())
        finally:
            conn.close()

    def do_GET(self):     self._proxy()
    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        self._proxy(self.rfile.read(length) if length else None)
    def do_PUT(self):
        length = int(self.headers.get("Content-Length", 0))
        self._proxy(self.rfile.read(length) if length else None)
    def do_DELETE(self):  self._proxy()
    def do_OPTIONS(self):
        self.send_response(200)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, PUT, DELETE, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "*")
        self.end_headers()
    def log_message(self, *args): pass

server = http.server.HTTPServer(('0.0.0.0', 5001), Proxy)
server.serve_forever()
"""

with open("proxy.py", "w") as f:
    f.write(proxy_code)

subprocess.Popen(["python", "proxy.py"],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("Proxy started on port 5001")
time.sleep(3)

# ── 4. Connect ngrok to proxy on port 5001 ────────────────────────
ngrok.set_auth_token(userdata.get('NGROK_AUTH_TOKEN'))

# Replace with your custom domain
public_url = ngrok.connect(5001, domain="ia-mlflow-dashboard.ngrok.app")
print(f"\nMLflow UI: {public_url}\n")

Waiting for MLflow...


MLflow ready
Proxy started on port 5001

MLflow UI: NgrokTunnel: "https://ia-mlflow-dashboard.ngrok.app" -> "http://localhost:5001"



Trace(trace_id=tr-d5ed4c47fed2fd7d60ac567ec82a2737)

###Testing Run the notebook once, go to ngrok and stop agent and run the MLflow dashboard cell again. If ngrok agent has 3 active session, stop agent my subscription only allow 3 active agent

In [150]:
import mlflow
import mlflow.langchain
import mlflow.openai
import os
import openai
import json

# Set the MLflow tracking URI to connect to the server started by ngrok
# The ngrok setup usually exposes localhost:5000
mlflow.set_tracking_uri("http://localhost:5000")

# Set an experiment name for better organization in the MLflow UI
mlflow.set_experiment("RAG Chatbot Interactions")
mlflow.openai.autolog()

print("MLflow tracking setup complete.")

MLflow tracking setup complete.


In [130]:
import requests

runs = requests.post(
    "http://localhost:5001/api/2.0/mlflow/runs/search",
    json={"experiment_ids": ["1"], "max_results": 50}
).json()

print(f"Number of runs: {len(runs.get('runs', []))}")
for run in runs.get('runs', []):
    info = run['info']
    metrics = run.get('data', {}).get('metrics', [])
    print(f"\nRun ID: {info['run_id']}")
    print(f"Status: {info['status']}")
    print(f"Metrics: {metrics}")

Number of runs: 0


In [131]:
import requests

# Check if MLflow server is still responding
try:
    r = requests.get("http://localhost:5001/health", timeout=5)
    print("Health:", r.status_code, r.text)
except Exception as e:
    print("Server down:", e)

# Check if runs are still in DB
try:
    r = requests.post(
        "http://localhost:5001/api/2.0/mlflow/runs/search",
        json={"experiment_ids": ["1"], "max_results": 50}
    ).json()
    print("Runs found:", len(r.get("runs", [])))
except Exception as e:
    print("Runs search failed:", e)

Health: 200 OK
Runs found: 0


In [132]:
# Check if the server is actually responding
!curl -X POST http://localhost:5001/api/2.0/mlflow/experiments/search \
  -H "Content-Type: application/json" \
  -d '{"max_results": 100}'

{
  "experiments": [
    {
      "experiment_id": "1",
      "name": "RAG Chatbot Interactions",
      "artifact_location": "mlflow-artifacts:/1",
      "lifecycle_stage": "active",
      "last_update_time": 1781965696255,
      "creation_time": 1781965696255,
      "workspace": "default"
    },
    {
      "experiment_id": "0",
      "name": "Default",
      "artifact_location": "mlflow-artifacts:/0",
      "lifecycle_stage": "active",
      "last_update_time": 1781965648616,
      "creation_time": 1781965648616,
      "workspace": "default"
    }
  ]
}

Check specific runs

In [133]:
import requests

runs = requests.post(
    "http://localhost:5001/api/2.0/mlflow/runs/search",
    json={"experiment_ids": ["1"], "max_results": 50}
).json()

for run in runs.get("runs", []):
    info = run["info"]
    metrics = run.get("data", {}).get("metrics", [])
    params  = run.get("data", {}).get("params", [])
    print(f"Run: {info['run_name']} | ID: {info['run_id']} | Metrics: {len(metrics)} | Params: {len(params)}")

In [146]:
import subprocess, time

subprocess.run("pkill -f 'mlflow server'", shell=True)
time.sleep(3)

# Find where mlruns actually lives
result = subprocess.run("find /content -name 'mlruns' -type d 2>/dev/null",
                        shell=True, capture_output=True, text=True)
print("mlruns found at:", result.stdout.strip())

mlruns found at: 


Check artifacts location

In [135]:
'''import requests

runs = requests.post(
    "http://localhost:5001/api/2.0/mlflow/runs/search",
    json={"experiment_ids": ["1"], "max_results": 50}
).json()

for run in runs.get("runs", []):
    info = run["info"]
    print(f"{info['run_name']} | artifact_uri: {info['artifact_uri']}")'''

'import requests\n\nruns = requests.post(\n    "http://localhost:5001/api/2.0/mlflow/runs/search",\n    json={"experiment_ids": ["1"], "max_results": 50}\n).json()\n\nfor run in runs.get("runs", []):\n    info = run["info"]\n    print(f"{info[\'run_name\']} | artifact_uri: {info[\'artifact_uri\']}")'

#Using OpenAI gpt-4 model

In [151]:
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

## Embeddding

In [152]:
%%time

# Use PyPDFLoader to correctly load the content from the PDF.
# This ensures the PDF is parsed and its text extracted, avoiding UnicodeDecodeError.
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(dataset)
docs = loader.load()
# The PDF has already been loaded into the 'docs' variable using PyPDFLoader in a previous cell.
# Assign the loaded documents to medical_chatbot for subsequent processing.

# 2. Setting up Embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("Setup complete. Your chatbot is ready for testing, Let's Go Isaac!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Setup complete. Your chatbot is ready for testing, Let's Go Isaac!
CPU times: user 6.02 s, sys: 1.14 s, total: 7.16 s
Wall time: 9.97 s


# Vector Embedding to convert string to binary

*   Temperature improves novel response
*   Max tokens



In [153]:
%%time
llm = ChatOpenAI(model="gpt-4-turbo", temperature=0.8, max_tokens=250)

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

CPU times: user 184 ms, sys: 17.1 ms, total: 201 ms
Wall time: 213 ms


In [154]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

#Chunking

In [155]:
%%time
text_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
all_splits = text_splitter.split_documents(docs)

CPU times: user 25.4 ms, sys: 3.25 ms, total: 28.6 ms
Wall time: 30.8 ms


#FAISS from Chunks



*   Similarity searching
*   Clustering of vectors



In [157]:
%%time
# Create FAISS index from chunks
db = FAISS.from_documents(all_splits, embeddings)

CPU times: user 4.87 s, sys: 458 ms, total: 5.33 s
Wall time: 7.73 s


# CHAT WORKS AUDIO DOESNT


In [142]:
import openai
import os # Ensure os is imported for environment variable access
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chatbot_prompt = ChatPromptTemplate.from_messages([
     ("system", """You are an a public health official with the goal of finding what drives global longevity. You need to answer questions related to some world data bank in a  polite manner to a prospective public officials\n\n         Follow these rules:\n1. Always answer based on the provided context only. Do NOT make up an answer.\n2. If unsure, say \"I don't have enough information to answer that\"\n3. Dealing with strict confidential information divulge only information asked\n4. Use bullet points if needed\n
      Context:\n# {context}"""),
     ("human", "Question: {question}"),
     # Removed: ("ai", "Answer: "), to prevent incomplete AI responses
])

# New prompt for rephrasing the question using chat history
condense_question_prompt = ChatPromptTemplate.from_messages([
     ("system", "Given a chat history and a follow-up question, rephrase the follow-up question to be a standalone question."),
     MessagesPlaceholder(variable_name="chat_history"),
     ("human", "{question}"),
])

def ask_question(question: str, history=None) -> str:
    chat_history_for_llm = []

    if history:
        for turn in history:
            # Safely handle dictionary format, Gradio message objects, or legacy tuples
            if isinstance(turn, dict):
                role = turn.get("role")
                content = turn.get("content", "")
                if role in ["user", "human"]:
                    chat_history_for_llm.append(HumanMessage(content=content))
                elif role in ["assistant", "ai", "model"]:
                    chat_history_for_llm.append(AIMessage(content=content))
            elif hasattr(turn, "role"):  # If it arrives as a Gradio ChatMessage object
                if turn.role in ["user", "human"]:
                    chat_history_for_llm.append(HumanMessage(content=turn.content))
                elif turn.role in ["assistant", "ai", "model"]:
                    chat_history_for_llm.append(AIMessage(content=turn.content))
            elif isinstance(turn, (list, tuple)) and len(turn) == 2:
                if turn[0]: chat_history_for_llm.append(HumanMessage(content=turn[0]))
                if turn[1]: chat_history_for_llm.append(AIMessage(content=turn[1]))

    standalone_question = question
    if chat_history_for_llm:
        try:
            # Condense follow-up question using chat history
            condense_chain = condense_question_prompt | llm
            standalone_question = condense_chain.invoke({
                "question": question,
                "chat_history": chat_history_for_llm
            }).content
        except Exception as e:
            print(f"Error condensing question: {e}")
            standalone_question = question

    # Retrieve matching vector documents
    try:
        retrieved_docs = db.similarity_search(standalone_question, k=4)
    except NameError:
        retrieved_docs = vector_store.similarity_search(standalone_question, k=4)

    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs if hasattr(doc, 'page_content'))

    # Format the prompt messages
    messages = chatbot_prompt.format_messages(question=standalone_question, context=docs_content)

    # Convert LangChain message formats to raw dictionary maps to prevent SDK validation rejections
    formatted_payload = []
    for msg in messages:
        # Standardize roles into API compatible tags
        role_type = "user" if msg.type == "human" else "system" if msg.type == "system" else "assistant"
        formatted_payload.append({"role": role_type, "content": msg.content})

    # Invoke your model using the safe dictionary array payload
    return llm.invoke(formatted_payload).content


def predict(input, history):
     # This 'predict' function is not used by the multimodal_app or agent_predict, but leaving it as is.
     output = ask_question(input, history)

     history.append((input, output))

     return history, history


# This converts voice to text using OpenAI Whisper
def transcribe_audio(audio_path):
    # Ensure OPENAI_API_KEY is set in environment or Colab secrets
    if not os.environ.get("OPENAI_API_KEY"):
        return "Error: OPENAI_API_KEY environment variable not set."

    client = openai.OpenAI()
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file
        )
    return transcript.text


# This function will handle both text and transcribed voice input
# and manage the chat history for gr.Chatbot. It uses the globally defined ask_question.
# It now expects and returns history in the Gradio Chatbot format (list of lists/tuples).
def predict_for_multimodal(message, history):
    if history is None:
        history = []

    # 1. Compute the text generation answer
    rag_answer = ask_question(message, history)

    # 2. Append history using explicit dictionary format definitions to guarantee Gradio matching
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": rag_answer})

    return history, ""

# This function wraps transcribe_audio and then calls the main predict_for_multimodal
def voice_chat_handler(audio_path, history):
    if audio_path is None:
        return history or [], "" # Return history and clear audio input

    user_text = transcribe_audio(audio_path)
    if user_text.startswith("Error:"):
        if history is None:
            history = []
        history = list(history)
        history.append([None, user_text]) # Append error as bot message in standard Gradio format
        return history, "" # Return history and clear audio input

    # Call the main prediction function with the transcribed text
    return predict_for_multimodal(user_text, history)


with gr.Blocks() as multimodal_app:
    gr.Markdown("## Multimodal RAG Assistant")

    chatbot = gr.Chatbot(label="Chat History")

    with gr.Row():
        msg = gr.Textbox(placeholder="Type a question...", scale=4)
        audio_in = gr.Audio(sources="microphone", type="filepath", scale=1)

    msg.submit(
        predict_for_multimodal,
        [msg, chatbot],
        [chatbot, msg] # Clear the text input after submission
    )

    audio_in.stop_recording(
        voice_chat_handler,
        [audio_in, chatbot],
        [chatbot, audio_in] # Clear the audio input after submission
    )

multimodal_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6e322d2b1949eec0ad.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Chat and Audio Works

In [143]:
import openai
import os # Ensure os is imported for environment variable access
import asyncio # Added for async TTS
import edge_tts # Added for TTS
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chatbot_prompt = ChatPromptTemplate.from_messages([
     ("system", """You are an a public health official with the goal of finding what drives global longevity. You need to answer questions related to some world data bank in a  polite manner to a prospective public officials\n\n         Follow these rules:\n1. Always answer based on the provided context only. Do NOT make up an answer.\n2. If unsure, say \"I don't have enough information to answer that\"\n3. Dealing with strict confidential information divulge only information asked\n4. Use bullet points if needed\n
      Context:\n# {context}"""),
     ("human", "Question: {question}"),
     # Removed: ("ai", "Answer: "), to prevent incomplete AI responses
])

# New prompt for rephrasing the question using chat history
condense_question_prompt = ChatPromptTemplate.from_messages([
     ("system", "Given a chat history and a follow-up question, rephrase the follow-up question to be a standalone question."),
     MessagesPlaceholder(variable_name="chat_history"),
     ("human", "{question}"),
])

def ask_question(question: str, history=None) -> str:
    chat_history_for_llm = []

    if history:
        for turn in history:
            # Safely handle dictionary format, Gradio message objects, or legacy tuples
            if isinstance(turn, dict):
                role = turn.get("role")
                content = turn.get("content", "")
                if role in ["user", "human"]:
                    chat_history_for_llm.append(HumanMessage(content=content))
                elif role in ["assistant", "ai", "model"]:
                    chat_history_for_llm.append(AIMessage(content=content))
            elif hasattr(turn, "role"):  # If it arrives as a Gradio ChatMessage object
                if turn.role in ["user", "human"]:
                    chat_history_for_llm.append(HumanMessage(content=turn.content))
                elif turn.role in ["assistant", "ai", "model"]:
                    chat_history_for_llm.append(AIMessage(content=turn.content))
            elif isinstance(turn, (list, tuple)) and len(turn) == 2:
                if turn[0]: chat_history_for_llm.append(HumanMessage(content=turn[0]))
                if turn[1]: chat_history_for_llm.append(AIMessage(content=turn[1]))

    standalone_question = question
    if chat_history_for_llm:
        try:
            # Condense follow-up question using chat history
            condense_chain = condense_question_prompt | llm
            standalone_question = condense_chain.invoke({
                "question": question,
                "chat_history": chat_history_for_llm
            }).content
        except Exception as e:
            print(f"Error condensing question: {e}")
            standalone_question = question

    # Retrieve matching vector documents
    try:
        retrieved_docs = db.similarity_search(standalone_question, k=4)
    except NameError:
        retrieved_docs = vector_store.similarity_search(standalone_question, k=4)

    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs if hasattr(doc, 'page_content'))

    # Format the prompt messages
    messages = chatbot_prompt.format_messages(question=standalone_question, context=docs_content)

    # Convert LangChain message formats to raw dictionary maps to prevent SDK validation rejections
    formatted_payload = []
    for msg in messages:
        # Standardize roles into API compatible tags
        role_type = "user" if msg.type == "human" else "system" if msg.type == "system" else "assistant"
        formatted_payload.append({"role": role_type, "content": msg.content})

    # Invoke your model using the safe dictionary array payload
    return llm.invoke(formatted_payload).content


def predict(input, history):
     # This 'predict' function is not used by the multimodal_app or agent_predict, but leaving it as is.
     output = ask_question(input, history)

     history.append((input, output))

     return history, history


# This converts voice to text using OpenAI Whisper
def transcribe_audio(audio_path):
    # Ensure OPENAI_API_KEY is set in environment or Colab secrets
    if not os.environ.get("OPENAI_API_KEY"):
        return "Error: OPENAI_API_KEY environment variable not set."

    client = openai.OpenAI()
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file
        )
    return transcript.text

# Added text-to-speech function
async def _tts(text):
    communicate = edge_tts.Communicate(text, "en-US-AndrewNeural")
    path = "response.mp3"
    await communicate.save(path)
    return path


# This function will handle both text and transcribed voice input
# and manage the chat history for gr.Chatbot. It uses the globally defined ask_question.
# It now expects and returns history in the Gradio Chatbot format (list of lists/tuples).
def predict_for_multimodal(message, history):
    if history is None:
        history = []

    # 1. Compute the text generation answer
    rag_answer = mlflow_tracked_ask_question(message, history)

    # 2. Append history using explicit dictionary format definitions to guarantee Gradio matching
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": rag_answer})

    return history, ""

# This function wraps transcribe_audio and then calls the main predict_for_multimodal and includes TTS
def voice_chat_handler(audio_path, history):
    status_msg = "" # For the status_box
    audio_output_path = None
    user_text = "" # Initialize user_text
    bot_text = "" # Initialize bot_text

    if history is None:
        history = []

    if audio_path is None:
        status_msg = "No audio input received. Please record your question."
        return history, audio_output_path, status_msg

    try:
        # 1. Transcribe audio
        status_msg = "Converting audio to text..."
        user_text = transcribe_audio(audio_path)
        if user_text.startswith("Error:"):
            status_msg = user_text
            history.append({"role": "assistant", "content": status_msg})
            return history, audio_output_path, status_msg

        # 2. Get RAG answer
        status_msg = f"User: {user_text}\nGenerating response..."
        bot_text = mlflow_tracked_ask_question(user_text, history) # Use mlflow_tracked_ask_question here

        # 3. Convert bot's response to audio
        status_msg = "Converting response to speech..."
        audio_output_path = asyncio.run(_tts(bot_text))

        status_msg = "Response generated successfully!"

    except Exception as e:
        status_msg = f"Error in voice processing: {e}"
        bot_text = f"I encountered an error while processing your request: {e}"
        audio_output_path = None

    # 4. Update history
    history.append({"role": "user", "content": user_text})
    history.append({"role": "assistant", "content": bot_text})

    return history, audio_output_path, status_msg


with gr.Blocks() as multimodal_app:
    gr.Markdown("## Multimodal RAG Assistant")

    chatbot = gr.Chatbot(label="Chat History")
    # Added status_box and audio_out components
    status_box = gr.Textbox(label="Voice Status", interactive=False, visible=True)
    audio_out = gr.Audio(label="🔊 Voice Response", autoplay=True, streaming=True) # Added for TTS output

    with gr.Row():
        msg = gr.Textbox(placeholder="Type a question...", scale=4)
        audio_in = gr.Audio(sources="microphone", type="filepath", scale=1)

    msg.submit(
        predict_for_multimodal,
        [msg, chatbot],
        [chatbot, msg] # Clear the text input after submission
    )

    audio_in.stop_recording(
        voice_chat_handler,
        [audio_in, chatbot],
        [chatbot, audio_out, status_box] # Updated outputs for voice_chat_handler
    )

multimodal_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bfa8d24da4f842be1e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Trace(trace_id=tr-0a499df9aede8229b0cc951c611259f9)